In [6]:
import anthropic
import json
import os

client = anthropic.Anthropic()
model = "claude-opus-4-6"

In [7]:
def get_text_edit_schema(model):
    if model.startswith("claude-3-7-sonnet"):
        return {"type": "text_editor_20250124", "name": "str_replace_editor"}
    elif model.startswith("claude-3-5-sonnet"):
        return {"type": "text_editor_20241022", "name": "str_replace_editor"}
    elif model.startswith("claude-opus-4") or model.startswith("claude-sonnet-4"):
        return {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}

In [8]:
def handle_text_editor(tool_input):
    command = tool_input["command"]
    
    if command == "view":
        path = tool_input["path"]
        if os.path.isdir(path):
            return "\n".join(os.listdir(path))
        with open(path, "r") as f:
            lines = f.readlines()
        if "view_range" in tool_input:
            start, end = tool_input["view_range"]
            lines = lines[start-1:end]
        return "".join(lines)

    elif command == "create":
        with open(tool_input["path"], "w") as f:
            f.write(tool_input.get("file_text", ""))
        return "File created."

    elif command == "str_replace":
        with open(tool_input["path"], "r") as f:
            content = f.read()
        content = content.replace(tool_input["old_str"], tool_input["new_str"], 1)
        with open(tool_input["path"], "w") as f:
            f.write(content)
        return "Replacement done."

    elif command == "insert":
        with open(tool_input["path"], "r") as f:
            lines = f.readlines()
        lines.insert(tool_input["insert_line"], tool_input["new_str"] + "\n")
        with open(tool_input["path"], "w") as f:
            f.writelines(lines)
        return "Text inserted."

In [9]:
def run_conversation(messages):
    schema = get_text_edit_schema(model)
    while True:
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            tools=[schema],
            messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        for block in response.content:
            if block.type == "text":
                print(block.text)

        if response.stop_reason != "tool_use":
            break

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                output = handle_text_editor(block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                    "is_error": False
                })
        messages.append({"role": "user", "content": tool_results})

    return messages

In [10]:
messages = [{"role": "user", "content": "Create a ./pi.py file with a function that calculates pi to the 5th digit. Then create ./test.py to test it."}]
run_conversation(messages)



Let me create both files for you.
Both files have been created. Here's a summary:

### `pi.py`
- Contains a `calculate_pi(precision=5)` function that computes pi using the **Leibniz series** (`π/4 = 1 - 1/3 + 1/5 - 1/7 + ...`).
- Uses 10 million iterations to ensure accuracy to at least 5 decimal places.
- Returns pi rounded to the specified precision (default: 5 digits → `3.14159`).

### `test.py`
- Uses Python's `unittest` framework with 5 test cases:
  - **`test_pi_to_5_decimal_places`** — verifies the result equals `3.14159`
  - **`test_pi_to_2_decimal_places`** — verifies rounding to 2 decimals gives `3.14`
  - **`test_pi_to_0_decimal_places`** — verifies rounding to 0 decimals gives `3.0`
  - **`test_default_precision`** — verifies the default argument works correctly
  - **`test_return_type_is_float`** — verifies the return type is `float`

You can run the tests with:
```bash
python test.py
```


[{'role': 'user',
  'content': 'Create a ./pi.py file with a function that calculates pi to the 5th digit. Then create ./test.py to test it.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='\n\nLet me create both files for you.', type='text'),
   ToolUseBlock(id='toolu_01K1bzW2Pm8KRZM53W8TV9wu', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': './pi.py', 'file_text': '"""Module to calculate pi to the 5th decimal digit using the Leibniz series."""\n\n\ndef calculate_pi(precision=5):\n    """Calculate pi to the specified number of decimal digits using the Leibniz series.\n\n    The Leibniz formula: pi/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 - ...\n\n    Args:\n        precision: Number of decimal digits to round to (default 5).\n\n    Returns:\n        Pi rounded to the given number of decimal digits.\n    """\n    pi = 0.0\n    sign = 1\n    denominator = 1\n    # The Leibniz series converges slowly, so we use a large number of iterations\n    for _ i